# Minimal Fig. 3(b) Reproduction With `bloqade-decoders`

This notebook keeps the MSD reproduction flow compact by delegating setup, training, evaluation, and plotting to `demo.msd_utils.experiment`.

Typical edit points:
- `prep_config` for injected/distilled magic-state preparation noise
- `run_config` for shot counts, uncertainty backend, and MLE threshold policy
- `plot_config` for display choices like minimum accepted fraction


In [ ]:
from dataclasses import replace
import sys
from pathlib import Path

# NOTE: all this is because bloqade-decoders is not a dependency in bloqade-lanes, and we're currently developing in bloqade-lanes.
PROJECT_ROOT_CANDIDATES = [Path.cwd(), Path.cwd().parent]
for candidate in PROJECT_ROOT_CANDIDATES:
    candidate = candidate.resolve()
    if (candidate / 'demo' / 'msd_utils').exists():
        sys.path.insert(0, str(candidate))
        break
else:
    raise FileNotFoundError('Could not locate repo root containing demo/msd_utils.')

LOCAL_DECODER_SRC_CANDIDATES = [
    Path.cwd() / '..' / 'bloqade-decoders' / 'src',
    Path.cwd() / 'bloqade-decoders' / 'src',
    Path.cwd().parent / 'bloqade-decoders' / 'src',
    Path.cwd().parent.parent / 'bloqade-decoders' / 'src',
]
for candidate in LOCAL_DECODER_SRC_CANDIDATES:
    candidate = candidate.resolve()
    if candidate.exists():
        sys.path.insert(0, str(candidate))
        break
else:
    raise FileNotFoundError('Could not locate sibling bloqade-decoders/src checkout.')

from bloqade.decoders import GurobiDecoder, TableDecoder
from demo.msd_utils.experiment import (
    MSDPlotConfig,
    MSDPrepConfig,
    MSDRunConfig,
    plot_msd_fig3b,
    print_msd_fig3b_summary,
    run_msd_fig3b_experiment,
)


In [ ]:
prep_config = MSDPrepConfig(
    theta_offset=0.30,
    phi_offset=0.0,
    lam_offset=0.0,
)

run_config = replace(
    MSDRunConfig.paper(),
    mld_train_shots=10_000_000,
    eval_shots=1_000_000,
    uncertainty_backend='wilson',
    mle_threshold_policy='quantile',
)

plot_config = MSDPlotConfig(min_accepted_fraction=0.02)


In [ ]:
result = run_msd_fig3b_experiment(
    prep_config=prep_config,
    run_config=run_config,
    plot_config=plot_config,
    table_decoder_cls=TableDecoder,
    gurobi_decoder_cls=GurobiDecoder,
    verbose=True,
)

print_msd_fig3b_summary(result)


In [ ]:
plot_msd_fig3b(result)
